In [ ]:
!pip install -q edge-tts resemblyzer librosa soundfile scipy pandas openpyxl nest_asyncio

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Configuration
AUDIO_BASE_DIR = "/content/drive/MyDrive/UrduSER A Dataset for Urdu Speech Emotion Recognition/UrduSER A Dataset for Urdu Speech Emotion Recognition"
EXCEL_PATH = "/content/drive/MyDrive/UrduSER A Dataset for Urdu Speech Emotion Recognition/UrduSER A Dataset for Urdu Speech Emotion Recognition/UrSEC_Description.xlsx"

EDGE_VOICE = "ur-PK-AsadNeural"   # male Urdu voice
OUTPUT_ROOT = "/content/edgetts_urduser_eval"
REFERENCE_DIR = f"{OUTPUT_ROOT}/reference"
GENERATED_DIR = f"{OUTPUT_ROOT}/generated_edgetts"
ANCHOR_DIR = f"{OUTPUT_ROOT}/anchor"
RESULTS_CSV = f"{OUTPUT_ROOT}/urdu_edgetts_evaluation.csv"
COMBINED_ZIP = f"{OUTPUT_ROOT}/edgetts_all_outputs.zip"

RESET_OUTPUTS = True

In [6]:
import os
import io
import shutil
import zipfile
import asyncio
from pathlib import Path

import edge_tts
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from scipy.signal import resample
from resemblyzer import VoiceEncoder, preprocess_wav
import nest_asyncio

nest_asyncio.apply()

encoder = VoiceEncoder()

os.makedirs(OUTPUT_ROOT, exist_ok=True)

if RESET_OUTPUTS:
    for d in [REFERENCE_DIR, GENERATED_DIR, ANCHOR_DIR]:
        if os.path.exists(d):
            shutil.rmtree(d)

for d in [REFERENCE_DIR, GENERATED_DIR, ANCHOR_DIR]:
    os.makedirs(d, exist_ok=True)

print("Output root:", OUTPUT_ROOT)
print("Reference dir:", REFERENCE_DIR)
print("Generated dir:", GENERATED_DIR)
print("Anchor dir:", ANCHOR_DIR)

Loaded the voice encoder model on cuda in 0.42 seconds.
Output root: /content/edgetts_urduser_eval
Reference dir: /content/edgetts_urduser_eval/reference
Generated dir: /content/edgetts_urduser_eval/generated_edgetts
Anchor dir: /content/edgetts_urduser_eval/anchor


In [9]:
import pandas as pd

df = pd.read_excel(EXCEL_PATH)
print(df.head())

                                UrSEC Description Unnamed: 1 Unnamed: 2  \
0                                           تشریح      جذبات        صنف   
1  اور تم سارے شہر میں یہ ڈھنڈورا پیٹتی آ رہی ہو         غصہ        مرد   
2    تم نہیں سمجھ رہی ہو تمہیں سمجھ نہیں آ رہی ہے        غصہ        مرد   
3                      نکلو یہاں سے باہر، چلو جاؤ        غصہ        مرد   
4                              بکواس کر رہے ہو تم        غصہ        مرد   

   Unnamed: 3             Unnamed: 4    Unnamed: 5    Unnamed: 6 Unnamed: 7  \
0      اداکار  اداکار  کاآئی ڈی نمبر  نمونہ کی شکل  نمونہ کی شرح     فارمیٹ   
1  محمود اسلم                      1  32-Bit Float      41100 Hz        WAV   
2  محمود اسلم                      1  32-Bit Float      41100 Hz        WAV   
3  محمود اسلم                      1  32-Bit Float      41100 Hz        WAV   
4  محمود اسلم                      1  32-Bit Float      41100 Hz        WAV   

            Unnamed: 8   Unnamed: 9 Unnamed: 10  
0  دورانیہ سیکنڈز میں   

In [10]:
print(df.iloc[0])

UrSEC Description                    تشریح
Unnamed: 1                           جذبات
Unnamed: 2                             صنف
Unnamed: 3                          اداکار
Unnamed: 4           اداکار  کاآئی ڈی نمبر
Unnamed: 5                    نمونہ کی شکل
Unnamed: 6                    نمونہ کی شرح
Unnamed: 7                          فارمیٹ
Unnamed: 8             دورانیہ سیکنڈز میں 
Unnamed: 9                     فائل کا نام
Unnamed: 10                     سیریل نمبر
Name: 0, dtype: object


In [11]:
import pandas as pd

df = pd.read_excel(EXCEL_PATH, header=0)

# assign correct Urdu headers manually
df.columns = [
    "تشریح",
    "جذبات",
    "صنف",
    "اداکار",
    "اداکار_آئی_ڈی",
    "نمونہ_کی_شکل",
    "نمونہ_کی_شرح",
    "فارمیٹ",
    "دورانیہ_سیکنڈز",
    "فائل_کا_نام",
    "سیریل_نمبر"
]
print(df.head())

                                            تشریح  جذبات  صنف      اداکار  \
0                                           تشریح  جذبات  صنف      اداکار   
1  اور تم سارے شہر میں یہ ڈھنڈورا پیٹتی آ رہی ہو     غصہ  مرد  محمود اسلم   
2    تم نہیں سمجھ رہی ہو تمہیں سمجھ نہیں آ رہی ہے    غصہ  مرد  محمود اسلم   
3                      نکلو یہاں سے باہر، چلو جاؤ    غصہ  مرد  محمود اسلم   
4                              بکواس کر رہے ہو تم    غصہ  مرد  محمود اسلم   

           اداکار_آئی_ڈی  نمونہ_کی_شکل  نمونہ_کی_شرح  فارمیٹ  \
0  اداکار  کاآئی ڈی نمبر  نمونہ کی شکل  نمونہ کی شرح  فارمیٹ   
1                      1  32-Bit Float      41100 Hz     WAV   
2                      1  32-Bit Float      41100 Hz     WAV   
3                      1  32-Bit Float      41100 Hz     WAV   
4                      1  32-Bit Float      41100 Hz     WAV   

        دورانیہ_سیکنڈز  فائل_کا_نام  سیریل_نمبر  
0  دورانیہ سیکنڈز میں   فائل کا نام  سیریل نمبر  
1                    3     1_0_1_01           1  
2 

In [12]:
print(df.head())

                                            تشریح  جذبات  صنف      اداکار  \
0                                           تشریح  جذبات  صنف      اداکار   
1  اور تم سارے شہر میں یہ ڈھنڈورا پیٹتی آ رہی ہو     غصہ  مرد  محمود اسلم   
2    تم نہیں سمجھ رہی ہو تمہیں سمجھ نہیں آ رہی ہے    غصہ  مرد  محمود اسلم   
3                      نکلو یہاں سے باہر، چلو جاؤ    غصہ  مرد  محمود اسلم   
4                              بکواس کر رہے ہو تم    غصہ  مرد  محمود اسلم   

           اداکار_آئی_ڈی  نمونہ_کی_شکل  نمونہ_کی_شرح  فارمیٹ  \
0  اداکار  کاآئی ڈی نمبر  نمونہ کی شکل  نمونہ کی شرح  فارمیٹ   
1                      1  32-Bit Float      41100 Hz     WAV   
2                      1  32-Bit Float      41100 Hz     WAV   
3                      1  32-Bit Float      41100 Hz     WAV   
4                      1  32-Bit Float      41100 Hz     WAV   

        دورانیہ_سیکنڈز  فائل_کا_نام  سیریل_نمبر  
0  دورانیہ سیکنڈز میں   فائل کا نام  سیریل نمبر  
1                    3     1_0_1_01           1  
2 

In [13]:
df = df.rename(columns={
    "تشریح": "text",
    "جذبات": "emotion",
    "صنف": "gender",
    "اداکار": "actor",
    "اداکار_آئی_ڈی": "actor_id",
    "نمونہ_کی_شکل": "bit_depth",
    "نمونہ_کی_شرح": "sample_rate",
    "فارمیٹ": "format",
    "دورانیہ_سیکنڈز": "duration_sec",
    "فائل_کا_نام": "file_name",
    "سیریل_نمبر": "sr_no"
})

In [14]:
df.info()
df.head()
df["emotion"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3501 entries, 0 to 3500
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   text          3501 non-null   object
 1   emotion       3501 non-null   object
 2   gender        3501 non-null   object
 3   actor         3501 non-null   object
 4   actor_id      3501 non-null   object
 5   bit_depth     3501 non-null   object
 6   sample_rate   3501 non-null   object
 7   format        3501 non-null   object
 8   duration_sec  3501 non-null   object
 9   file_name     3501 non-null   object
 10  sr_no         3501 non-null   object
dtypes: object(11)
memory usage: 301.0+ KB


,count
emotion,
غصہ,500
خوف,500
بوریت,500
ناگواری,500
غیر جانبدار,500
خوشی,500
افسردہ,500
جذبات,1


In [45]:
ranges = {
    "غصہ": [1,2,3,4,5,6,7,8,9,19],
    "خوف": [63,64,65,66,67,68,69,70,71,72],
    "بوریت": [90,91,92,93,94,95,96,97,98,99],
    "ناگواری": [81,82,83,84,85,86,87,88,89,90],
    "خوشی": [64,65,66,67,68,69,70,71,72,73],
    "افسردہ": [54,55,56,57,60,61,62,63,64,65]
}

In [46]:
ranges = {
    emotion: [i - 1 for i in indices]
    for emotion, indices in ranges.items()
}

print(ranges)

{'غصہ': [0, 1, 2, 3, 4, 5, 6, 7, 8, 18], 'خوف': [62, 63, 64, 65, 66, 67, 68, 69, 70, 71], 'بوریت': [89, 90, 91, 92, 93, 94, 95, 96, 97, 98], 'ناگواری': [80, 81, 82, 83, 84, 85, 86, 87, 88, 89], 'خوشی': [63, 64, 65, 66, 67, 68, 69, 70, 71, 72], 'افسردہ': [53, 54, 55, 56, 59, 60, 61, 62, 63, 64]}


In [47]:

result = []
for emotion, indices in ranges.items():
    group = df[df["emotion"] == emotion].reset_index(drop=True)
    selected = group.iloc[indices].copy()
    result.append(selected)

final_df = pd.concat(result, ignore_index=True)

print("Total selected rows:", len(final_df))
display(final_df[["sr_no", "emotion", "file_name", "text"]].head(20))

Total selected rows: 60


,sr_no,emotion,file_name,text
0,1,غصہ,1_0_1_01,اور تم سارے شہر میں یہ ڈھنڈورا پیٹتی آ رہی ہو
1,2,غصہ,1_0_1_02,تم نہیں سمجھ رہی ہو تمہیں سمجھ نہیں آ رہی ہے
2,3,غصہ,1_0_1_03,نکلو یہاں سے باہر، چلو جاؤ
3,4,غصہ,1_0_1_04,بکواس کر رہے ہو تم
4,5,غصہ,1_0_1_05,اگر اسے بات کرنے کی تمیز نہیں ہے تو اسے اس گھر...
5,6,غصہ,1_0_1_06,احسان کے گھر میں پینٹ کا کام مکمل نہیں ہوا ابھ...
6,7,غصہ,1_0_1_07,زبیر بس بس، چلو یہاں سے
7,8,غصہ,1_0_1_08,بس بس میں آ گیا ہوں نا مرتسم خان، مجھ سے بات کرو
8,9,غصہ,1_0_1_09,میں نے دوستی کا ہاتھ بڑھایا تو ہاتھ تھام لیا ا...
9,19,غصہ,1_0_1_19,میری اجازت کے بغیر میری زمینوں میں گھسے اور می...


In [48]:
AUDIO_EXTENSIONS = (".wav", ".mp3", ".flac", ".ogg", ".m4a")

def normalize_filename(name):
    name = str(name).strip()
    return os.path.basename(name)

def build_audio_index(base_dir):
    audio_map = {}
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.lower().endswith(AUDIO_EXTENSIONS):
                audio_map.setdefault(file, os.path.join(root, file))
    return audio_map

audio_index = build_audio_index(AUDIO_BASE_DIR)
print(f"Indexed {len(audio_index)} audio files from dataset.")

def resolve_audio_path(file_name):
    raw_name = normalize_filename(file_name)
    candidates = [raw_name]

    stem, ext = os.path.splitext(raw_name)
    if ext == "":
        candidates.append(raw_name + ".wav")

    for cand in candidates:
        if cand in audio_index:
            return audio_index[cand]

    for cand in candidates:
        cand_stem = os.path.splitext(cand)[0].lower()
        for indexed_name, indexed_path in audio_index.items():
            if os.path.splitext(indexed_name)[0].lower() == cand_stem:
                return indexed_path

    return None

final_df["resolved_audio_path"] = final_df["file_name"].apply(resolve_audio_path)
final_df["reference_found"] = final_df["resolved_audio_path"].notna()

print("Reference files found:", int(final_df["reference_found"].sum()), "/", len(final_df))
missing_df = final_df[~final_df["reference_found"]][["sr_no", "emotion", "file_name"]]
display(missing_df.head(20))

Indexed 3500 audio files from dataset.
Reference files found: 60 / 60


,sr_no,emotion,file_name


In [49]:
missing_count = (~final_df["reference_found"]).sum()
if missing_count > 0:
    raise FileNotFoundError(
        f"{missing_count} selected reference audio files were not found in AUDIO_BASE_DIR. "
        "Check your Google Drive path and dataset folder structure."
    )
else:
    print("All selected reference audios were found successfully.")

All selected reference audios were found successfully.


In [50]:
display(final_df[["sr_no", "emotion", "file_name", "resolved_audio_path"]])

,sr_no,emotion,file_name,resolved_audio_path
0,1,غصہ,1_0_1_01,/content/drive/MyDrive/UrduSER A Dataset for U...
1,2,غصہ,1_0_1_02,/content/drive/MyDrive/UrduSER A Dataset for U...
2,3,غصہ,1_0_1_03,/content/drive/MyDrive/UrduSER A Dataset for U...
3,4,غصہ,1_0_1_04,/content/drive/MyDrive/UrduSER A Dataset for U...
4,5,غصہ,1_0_1_05,/content/drive/MyDrive/UrduSER A Dataset for U...
5,6,غصہ,1_0_1_06,/content/drive/MyDrive/UrduSER A Dataset for U...
6,7,غصہ,1_0_1_07,/content/drive/MyDrive/UrduSER A Dataset for U...
7,8,غصہ,1_0_1_08,/content/drive/MyDrive/UrduSER A Dataset for U...
8,9,غصہ,1_0_1_09,/content/drive/MyDrive/UrduSER A Dataset for U...
9,19,غصہ,1_0_1_19,/content/drive/MyDrive/UrduSER A Dataset for U...


In [51]:
async def tts_to_file(text, out_path, voice=EDGE_VOICE):
    communicate = edge_tts.Communicate(text=str(text), voice=voice)
    await communicate.save(out_path)

def synthesize_edgetts(text, out_path, voice=EDGE_VOICE):
    loop = asyncio.get_event_loop()
    if loop.is_running():
        loop.run_until_complete(tts_to_file(text, out_path, voice))
    else:
        loop.run_until_complete(tts_to_file(text, out_path, voice))

def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [52]:
results = []

for i, row in final_df.reset_index(drop=True).iterrows():
    text = row["text"]
    emotion = row["emotion"]
    file_name = row["file_name"]
    ref_source_path = row["resolved_audio_path"]
    sr_no = int(row["sr_no"])

    sample_id = f"urduser_{i+1:02d}"
    ref_out_path = os.path.join(REFERENCE_DIR, f"{sample_id}.wav")
    gen_out_path = os.path.join(GENERATED_DIR, f"{sample_id}.wav")
    anc_out_path = os.path.join(ANCHOR_DIR, f"{sample_id}.wav")

    # Load reference audio from dataset
    ref_speech, ref_sr = librosa.load(ref_source_path, sr=None, mono=True)

    # Save normalized reference copy
    sf.write(ref_out_path, ref_speech, ref_sr)

    # Generate EdgeTTS audio
    synthesize_edgetts(text, gen_out_path, EDGE_VOICE)
    gen_speech, gen_sr = librosa.load(gen_out_path, sr=None, mono=True)

    # Anchor creation
    downsampled = resample(ref_speech, int(len(ref_speech) * 4000 / ref_sr))
    upsampled = resample(downsampled, len(ref_speech))
    sf.write(anc_out_path, upsampled, ref_sr)

    # Embeddings + similarities
    ref_wav = preprocess_wav(ref_speech, ref_sr)
    gen_wav = preprocess_wav(gen_speech, gen_sr)
    anc_wav = preprocess_wav(upsampled, ref_sr)

    ref_emb = encoder.embed_utterance(ref_wav)
    gen_emb = encoder.embed_utterance(gen_wav)
    anc_emb = encoder.embed_utterance(anc_wav)

    ref_gen_similarity = cosine_similarity(ref_emb, gen_emb)
    ref_self_similarity = cosine_similarity(ref_emb, ref_emb)
    ref_anchor_similarity = cosine_similarity(ref_emb, anc_emb)
    anchor_gen_similarity = cosine_similarity(anc_emb, gen_emb)

    results.append({
        "dataset": "urduser",
        "sample_id": sample_id,
        "sr_no": sr_no,
        "emotion": emotion,
        "file_name": file_name,
        "reference_source_path": ref_source_path,
        "urdu_text": text,
        "audio_duration_sec": len(ref_speech) / ref_sr,
        "edge_voice": EDGE_VOICE,
        "ref_gen_similarity": ref_gen_similarity,
        "ref_self_similarity": ref_self_similarity,
        "ref_anchor_similarity": ref_anchor_similarity,
        "anchor_gen_similarity": anchor_gen_similarity
    })

    if (i + 1) % 10 == 0 or (i + 1) == len(final_df):
        print(f"Processed {i+1}/{len(final_df)} samples")

results_df = pd.DataFrame(results)
display(results_df.head())

Processed 10/60 samples
Processed 20/60 samples
Processed 30/60 samples
Processed 40/60 samples
Processed 50/60 samples
Processed 60/60 samples


,dataset,sample_id,sr_no,emotion,file_name,reference_source_path,urdu_text,audio_duration_sec,edge_voice,ref_gen_similarity,ref_self_similarity,ref_anchor_similarity,anchor_gen_similarity
0,urduser,urduser_01,1,غصہ,1_0_1_01,/content/drive/MyDrive/UrduSER A Dataset for U...,اور تم سارے شہر میں یہ ڈھنڈورا پیٹتی آ رہی ہو,3.181224,ur-PK-AsadNeural,0.617840,1.0,0.786524,0.530793
1,urduser,urduser_02,2,غصہ,1_0_1_02,/content/drive/MyDrive/UrduSER A Dataset for U...,تم نہیں سمجھ رہی ہو تمہیں سمجھ نہیں آ رہی ہے,2.414875,ur-PK-AsadNeural,0.504224,1.0,0.728345,0.539424
2,urduser,urduser_03,3,غصہ,1_0_1_03,/content/drive/MyDrive/UrduSER A Dataset for U...,نکلو یہاں سے باہر، چلو جاؤ,2.333605,ur-PK-AsadNeural,0.616204,1.0,0.787394,0.487528
3,urduser,urduser_04,4,غصہ,1_0_1_04,/content/drive/MyDrive/UrduSER A Dataset for U...,بکواس کر رہے ہو تم,1.474467,ur-PK-AsadNeural,0.596931,1.0,0.643764,0.471104
4,urduser,urduser_05,5,غصہ,1_0_1_05,/content/drive/MyDrive/UrduSER A Dataset for U...,اگر اسے بات کرنے کی تمیز نہیں ہے تو اسے اس گھر...,5.944286,ur-PK-AsadNeural,0.504778,1.0,0.692501,0.514293


In [53]:
# Summary statistics and CSV export
summary = results_df.groupby("dataset").agg({
    "ref_gen_similarity": "mean",
    "ref_self_similarity": "mean",
    "ref_anchor_similarity": "mean",
    "anchor_gen_similarity": "mean",
    "audio_duration_sec": "mean"
}).rename(columns={
    "ref_gen_similarity": "mean_gen_sim",
    "ref_self_similarity": "mean_self_sim",
    "ref_anchor_similarity": "mean_ref_anchor_sim",
    "anchor_gen_similarity": "mean_anchor_gen_sim",
    "audio_duration_sec": "mean_duration_sec"
})

summary["embedding_gap"] = summary["mean_self_sim"] - summary["mean_gen_sim"]

print("=== GLOBAL SUMMARY STATISTICS ===")
display(summary)

results_df.to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")
print("Saved CSV to:", RESULTS_CSV)

=== GLOBAL SUMMARY STATISTICS ===


,mean_gen_sim,mean_self_sim,mean_ref_anchor_sim,mean_anchor_gen_sim,mean_duration_sec,embedding_gap
dataset,,,,,,
urduser,0.563816,1.0,0.791976,0.540041,3.060687,0.436184


Saved CSV to: /content/edgetts_urduser_eval/urdu_edgetts_evaluation.csv


In [54]:
# Zip reference/, generated_edgetts/, and anchor/ together in one zip
def zip_multiple_folders(folders, zip_name):
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
        for folder in folders:
            for root, dirs, files in os.walk(folder):
                for file in files:
                    file_path = os.path.join(root, file)
                    arcname = os.path.join(os.path.basename(folder), os.path.relpath(file_path, folder))
                    zipf.write(file_path, arcname)

zip_multiple_folders(
    [REFERENCE_DIR, GENERATED_DIR, ANCHOR_DIR],
    COMBINED_ZIP
)

print("Created combined zip:", COMBINED_ZIP)

Created combined zip: /content/edgetts_urduser_eval/edgetts_all_outputs.zip


In [55]:
# Final outputs
print("Reference files:", len(os.listdir(REFERENCE_DIR)))
print("Generated files:", len(os.listdir(GENERATED_DIR)))
print("Anchor files:", len(os.listdir(ANCHOR_DIR)))
print("CSV exists:", os.path.exists(RESULTS_CSV))
print("ZIP exists:", os.path.exists(COMBINED_ZIP))

display(results_df[[
    "sample_id", "sr_no", "emotion", "file_name",
    "ref_gen_similarity", "ref_self_similarity",
    "ref_anchor_similarity", "anchor_gen_similarity"
]].head(20))

Reference files: 60
Generated files: 60
Anchor files: 60
CSV exists: True
ZIP exists: True


,sample_id,sr_no,emotion,file_name,ref_gen_similarity,ref_self_similarity,ref_anchor_similarity,anchor_gen_similarity
0,urduser_01,1,غصہ,1_0_1_01,0.617840,1.0,0.786524,0.530793
1,urduser_02,2,غصہ,1_0_1_02,0.504224,1.0,0.728345,0.539424
2,urduser_03,3,غصہ,1_0_1_03,0.616204,1.0,0.787394,0.487528
3,urduser_04,4,غصہ,1_0_1_04,0.596931,1.0,0.643764,0.471104
4,urduser_05,5,غصہ,1_0_1_05,0.504778,1.0,0.692501,0.514293
5,urduser_06,6,غصہ,1_0_1_06,0.564486,1.0,0.783519,0.590072
6,urduser_07,7,غصہ,1_0_1_07,0.584336,1.0,0.628910,0.558016
7,urduser_08,8,غصہ,1_0_1_08,0.553104,1.0,0.772470,0.507784
8,urduser_09,9,غصہ,1_0_1_09,0.608153,1.0,0.766937,0.562690
9,urduser_10,19,غصہ,1_0_1_19,0.509962,1.0,0.758960,0.472233
